# Demo: snow segmentation on Sentinel-2 imagery
Loads a trained model (`GASSL-basic` or `Unet`), runs it on one 13-band Sentinel-2 GeoTIFF and plots the snow probability map, the binary snow mask and the Dynamic World reference mask.

Run this notebook from the repository root.

In [ ]:
# Optional: install dependencies
# import sys
# !{sys.executable} -m pip install torch torchvision pytorch-lightning segmentation-models-pytorch rasterio opencv-python matplotlib

In [ ]:
import os
import sys
import json
from pathlib import Path

import cv2
import numpy as np
import rasterio
from rasterio.enums import Resampling
import torch
import matplotlib.pyplot as plt

# model code uses paths relative to 3_train_state_of_the_art/
REPO_ROOT = Path.cwd()
os.chdir(REPO_ROOT / "3_train_state_of_the_art")
sys.path.insert(0, os.getcwd())

from my_code.model import MyModel

## Settings

In [ ]:
ARCH = "GASSL-basic"  # "GASSL-basic" or "Unet"
MODEL_DIR = REPO_ROOT / "trained_model" / "all" / ARCH / "CrossEntropyLoss"
CKPT_PATH = MODEL_DIR / "snow-best.ckpt"
STATS_PATH = MODEL_DIR / "channel_stats.json"

INPUT_TIF = REPO_ROOT / "2_download_test_data_and_create_test_dataset" / "test_dataset" / "raw" / "POF_2020_28_07_2020.tif"
REFERENCE_TIF = REPO_ROOT / "2_download_test_data_and_create_test_dataset" / "test_dataset" / "snow_labels" / "POF_2020_28_07_2020.tif"  # optional, Dynamic World snow mask

IMAGE_SIZE = 512   # input is resized to this size (as in training)
MASK_SIZE = 1024   # model output size
THRESHOLD = 0.25   # snow probability threshold (best for GASSL in the paper; evaluated: 0.01, 0.1, 0.25, 0.5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Create model and load weights

In [ ]:
with open(STATS_PATH) as f:
    stats = json.load(f)
BANDS = stats["rgb_indexes"]  # all 13 Sentinel-2 bands (1-based raster indexes)
print("Bands:", BANDS)

model = MyModel(arch=ARCH,
                in_channels=len(BANDS),
                out_classes=2,
                train_channel_means=stats["means"],
                train_channel_stds=stats["stds"],
                loss_function="CrossEntropyLoss",
                n_iters_training=1)

ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["state_dict"], strict=True)
model = model.to(device)
model.eval()
print("Loaded:", CKPT_PATH)

## Load and preprocess input (same as training)

In [ ]:
with rasterio.open(INPUT_TIF) as src:
    image = src.read(BANDS, out_shape=(len(BANDS), IMAGE_SIZE, IMAGE_SIZE), resampling=Resampling.bilinear).astype(np.float32)

image = np.clip(image, 0, None)
image /= image.max()  # per-image max scaling, as in training

x = torch.from_numpy(image).unsqueeze(0).to(device)  # (1, 13, H, W)
print("Input:", x.shape)

## Run model

In [ ]:
with torch.no_grad():
    logits = model(x)  # (1, 2, 1024, 1024)
    snow_prob = torch.softmax(logits, dim=1)[0, 1].cpu().numpy()

snow_mask = (snow_prob >= THRESHOLD).astype(np.uint8)
print("Snow pixels:", snow_mask.sum(), f"({100 * snow_mask.mean():.2f} %)")

## Plot

In [ ]:
# true-color composite (B4, B3, B2) with percentile stretch for display
rgb = image[[3, 2, 1]].transpose(1, 2, 0)
lo, hi = np.percentile(rgb, (2, 98))
rgb = np.clip((rgb - lo) / (hi - lo + 1e-8), 0, 1)

panels = [("Sentinel-2 RGB", rgb, None),
          ("Snow probability", snow_prob, "viridis"),
          (f"Snow mask (p >= {THRESHOLD})", snow_mask, "gray")]

if REFERENCE_TIF.exists():
    with rasterio.open(REFERENCE_TIF) as src:
        gt = src.read(1).astype(np.uint8)
    gt = cv2.resize(gt, (MASK_SIZE, MASK_SIZE), interpolation=cv2.INTER_NEAREST)
    panels.append(("Reference (Dynamic World)", (gt > 0).astype(np.uint8), "gray"))

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
for ax, (title, img, cmap) in zip(axes, panels):
    ax.imshow(img, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Save output (optional)

In [ ]:
OUT_DIR = REPO_ROOT / "demo_output"
OUT_DIR.mkdir(exist_ok=True)
plt.imsave(OUT_DIR / f"{INPUT_TIF.stem}_{ARCH}_prob.png", snow_prob, cmap="gray", vmin=0, vmax=1)
plt.imsave(OUT_DIR / f"{INPUT_TIF.stem}_{ARCH}_mask_{THRESHOLD}.png", snow_mask, cmap="gray", vmin=0, vmax=1)
print("Saved to", OUT_DIR)